In [ ]:
from src.dfg_ms_plexus.training_setup import *

In [ ]:
# root = Path('F:/DATA/dfg_plexus/')
root = Path('/media/yannik/Intenso/DATA/dfg_plexus')

# df = pd.read_csv(root / "radiomics_features___combined.csv", delimiter=";")
# df = pd.read_csv(root / "radiomics_and_cnn_features___combined.csv", delimiter=";")
# df = pd.read_csv(root / "radiomics_features___combined_SA___normalized.csv", delimiter=";")
# df = pd.read_csv(root / "radiomics_features_shape_only___combined_SA___normalized_ICV.csv", delimiter="\t")
df = pd.read_csv(root / "radiomics_features___harmonized___combined___SA___normalized_ICV.csv", delimiter=";")

# Dropping disease_duration and lesion volume
ft = df.drop(columns=['label', 'patID', 'DWI (0no,1yes)', 'LesionVolume', 'DiseaseDuration', 'EDSS'])
label = df['label'].astype(int)
label, class_mapping = get_labels_hc_cis_ms(label)  # get_labels_hc_ms, get_labels_full
print(f"{df.shape=}")
print(f"{ft.shape=}")
print(f"{label.shape=}")

In [ ]:
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(label),
                                     y=label)

# Create a dictionary mapping class labels to their corresponding weights
class_weight_dict = dict(zip(np.unique(label), class_weights))
class_weight_dict

In [ ]:
seeds = [0, 1, 2, 3, 4]

all_test_reports = []
all_train_reports = []
all_test_cms = []
all_train_cms = []
all_best_params = []

labels_sorted = np.sort(np.unique(label))
class_names = list(class_mapping.keys())

X_train, X_test, y_train, y_test = train_test_split(
    ft,
    label,
    test_size=0.3,
    random_state=42,
    stratify=label,
)

n_classes = np.unique(y_train).size
is_multiclass = n_classes > 2

for seed in seeds:
    print(f"\n========== Seed {seed} ==========")

    pipeline = Pipeline([
        ("var", VarianceThreshold(0.001)),
        ('scaler', StandardScaler()),
        # ('pca', PCA(random_state=42)),
        # ('feature_selection', SelectKBest(score_func=f_classif)),
        ('classifier', LogisticRegression(random_state=seed))
    ])

    param_grid = {
         # 'pca__n_components': [0.95, 0.99, 0.999],
         # 'feature_selection__k': [10, 25, 50, 75, 'all'],
         'classifier__penalty': ['l2'],
         'classifier__solver': ['lbfgs'],
         'classifier__max_iter': [2000, 5000,],
         'classifier__C': [0.001, 0.01, 0.1, 1, 10],
         'classifier__class_weight': [None, class_weight_dict]
     }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scorer = make_scorer(f1_score, average='macro')

    grid_search = GridSearchCV(estimator=pipeline,
                               param_grid=param_grid,
                               cv=cv,
                               scoring=scorer,
                               n_jobs=12,
                               verbose=1,
                               refit=True)

    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    grid_search.fit(X_train, y_train, classifier__sample_weight=sample_weights)

    all_best_params.append(grid_search.best_params_)

    y_train_pred = grid_search.predict(X_train)
    y_test_pred = grid_search.predict(X_test)

    train_report = classification_report(
        y_train,
        y_train_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    test_report = classification_report(
        y_test,
        y_test_pred,
        labels=labels_sorted,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    all_train_reports.append(report_to_df(train_report))
    all_test_reports.append(report_to_df(test_report))

    all_train_cms.append(
        confusion_matrix(y_train, y_train_pred, labels=labels_sorted)
    )
    all_test_cms.append(
        confusion_matrix(y_test, y_test_pred, labels=labels_sorted)
    )

In [ ]:
train_summary = aggregate_reports(all_train_reports)
test_summary = aggregate_reports(all_test_reports)

print("\n--- Train Set: Mean ± Std over seeds ---")
display(train_summary)

print("\n--- Test Set: Mean ± Std over seeds ---")
display(test_summary)

In [ ]:
train_cms = np.asarray(all_train_cms)
test_cms = np.asarray(all_test_cms)

train_cms_norm = np.asarray([normalize_cm(cm) for cm in train_cms])
test_cms_norm = np.asarray([normalize_cm(cm) for cm in test_cms])

train_cm_mean = train_cms_norm.mean(axis=0)
train_cm_std = train_cms_norm.std(axis=0)

test_cm_mean = test_cms_norm.mean(axis=0)
test_cm_std = test_cms_norm.std(axis=0)

plot_mean_std_cm(
    train_cm_mean,
    train_cm_std,
    "Trainset Confusion Matrix: Mean ± Std over seeds",
    class_names,
)

plot_mean_std_cm(
    test_cm_mean,
    test_cm_std,
    "Testset Confusion Matrix: Mean ± Std over seeds",
    class_names,
)